# 01. Tìm hiểu dữ liệu và tiền xử lý

**Học phần:** Khai thác dữ liệu — Nhóm 12  
**Đề tài 16:** Phân nhóm quốc gia theo các yếu tố tạo nên mức độ hạnh phúc (World Happiness Report)

Notebook này gồm bốn phần: nêu bài toán và câu hỏi khai thác dữ liệu, tìm hiểu bộ dữ liệu World Happiness
Report giai đoạn 2015-2019, đánh giá chất lượng dữ liệu của từng năm, và dựng bảng dữ liệu trung gian
dùng chung cho các notebook sau.

Kết quả của notebook: tệp `data/interim/happiness_merged.csv` và hai bảng số liệu trong `reports/tables/`
(chất lượng dữ liệu theo năm, trích 10 dòng đầu của bảng trung gian).

In [1]:
from pathlib import Path
import sys

# Dò ngược lên để tìm gốc repo (thư mục chứa 'src') => notebook chạy đúng dù mở từ thư mục nào
here = Path.cwd().resolve()
project_root = next(p for p in [here, *here.parents] if (p / 'src').is_dir())
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

figures_dir = project_root / 'reports' / 'figures'
tables_dir = project_root / 'reports' / 'tables'
figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

print('Gốc repo:', project_root)

Gốc repo: C:\Users\xpaga\Documents\data_mining


## 1. Xác định bài toán và câu hỏi khai thác dữ liệu

1. **Bối cảnh và mục tiêu:** bài toán **gom cụm (clustering, học không giám sát)** nhằm phân nhóm các quốc gia theo mức độ phát triển kinh tế - xã hội - sức khỏe.
2. **Câu hỏi khai thác dữ liệu:**
   - Có thể phân chia các quốc gia thành những nhóm đặc trưng nào dựa trên 6 yếu tố cơ sở?
   - Các nhóm khác biệt rõ nhất ở những yếu tố nào?
   - Cấu trúc phân nhóm thay đổi thế nào qua 5 năm 2015-2019 và quốc gia nào chuyển nhóm?
   - Nhóm quốc gia nào ở mức thấp nhất và cần ưu tiên hỗ trợ?
3. **Quy tắc vàng của Đề tài 16:** tuyệt đối **KHÔNG** đưa `happiness_score` và `happiness_rank` vào ma trận đặc trưng dùng để gom cụm; hai thuộc tính này chỉ dùng để hậu kiểm ở notebook 04.
4. **Phạm vi thực nghiệm (theo ADR 0002):** gom cụm **độc lập cho từng năm** 2015-2019 rồi so sánh xu hướng, không gộp 5 năm thành một lần chạy.

## 2. Tìm hiểu dữ liệu
- **Nguồn:** Kaggle - World Happiness Report (5 tệp CSV theo năm 2015-2019).
- **Phương pháp thu thập:** khảo sát Gallup World Poll, câu hỏi Cantril Ladder (tự chấm cuộc sống từ 0 đến 10).
- **Đơn vị phân tích:** một quốc gia trong một năm.
- **6 yếu tố cơ sở dùng để gom cụm:** GDP per capita, Social support, Healthy life expectancy, Freedom, Generosity, Perceptions of corruption.

In [2]:
import pandas as pd
import numpy as np

from src.data.data_loader import (
    COUNTRY_NAME_FIXES,
    DEFAULT_YEARS,
    DROPPED_COLUMNS_NOTE,
    INTERIM_COLUMNS,
    build_interim_dataset,
    load_raw_data,
)
from src.features.feature_engineering import FEATURE_COLUMNS

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

raw_data = load_raw_data(raw_dir=project_root / 'data' / 'raw')
for year, df in raw_data.items():
    print(f'Năm {year}: {df.shape[0]} dòng, {df.shape[1]} cột')
print('\nCột của năm 2019 sau khi chuẩn hóa tên:')
print(list(raw_data[2019].columns))

Năm 2015: 158 dòng, 13 cột
Năm 2016: 157 dòng, 14 cột
Năm 2017: 155 dòng, 13 cột
Năm 2018: 156 dòng, 10 cột
Năm 2019: 156 dòng, 10 cột

Cột của năm 2019 sau khi chuẩn hóa tên:
['happiness_rank', 'country', 'happiness_score', 'gdp_per_capita', 'social_support', 'healthy_life_expectancy', 'freedom', 'generosity', 'corruption_perception', 'year']


## 3. Đánh giá chất lượng dữ liệu
Kiểm tra 4 khía cạnh: cỡ mẫu từng năm, cột chuẩn bị thiếu, giá trị thiếu ở 6 yếu tố cơ sở, và bản ghi trùng lặp theo tên quốc gia.

> Tên cột trong mã nguồn và trong tệp CSV đặt bằng **tiếng Anh** theo quy ước dự án; nội dung và nhận xét vẫn viết bằng tiếng Việt.
> Trong bảng dưới, giá trị **-1** ở các cột `missing_*` nghĩa là năm đó không có cột tương ứng (schema lệch giữa các năm), khác với giá trị **0** là có cột và không thiếu dữ liệu.

In [3]:
quality_records = []
for year, df in raw_data.items():
    missing_columns = [c for c in INTERIM_COLUMNS if c not in df.columns]
    record = {
        'year': year,
        'row_count': int(len(df)),
        'column_count_after_standardization': int(df.shape[1]),
        'missing_standard_column_count': len(missing_columns),
        'missing_standard_columns': ', '.join(missing_columns),
        'duplicate_country_rows': int(df['country'].duplicated().sum()),
    }
    for feature in FEATURE_COLUMNS:
        record[f'missing_{feature}'] = int(df[feature].isna().sum()) if feature in df.columns else -1
    quality_records.append(record)

quality_df = pd.DataFrame(quality_records)
quality_df.to_csv(tables_dir / 'data_quality_by_year.csv', index=False, encoding='utf-8-sig')
print('Đã lưu bảng chất lượng dữ liệu:', tables_dir / 'data_quality_by_year.csv')

quality_df.T

Đã lưu bảng chất lượng dữ liệu: C:\Users\xpaga\Documents\data_mining\reports\tables\data_quality_by_year.csv


,0,1,2,3,4
year,2015,2016,2017,2018,2019
row_count,158,157,155,156,156
column_count_after_standardization,13,14,13,10,10
missing_standard_column_count,0,0,1,2,2
missing_standard_columns,,,region,"region, dystopia_residual","region, dystopia_residual"
duplicate_country_rows,0,0,0,0,0
missing_gdp_per_capita,0,0,0,0,0
missing_social_support,0,0,0,0,0
missing_healthy_life_expectancy,0,0,0,0,0
missing_freedom,0,0,0,0,0


In [4]:
# Kiểm tra lệch tên quốc gia giữa các năm (nguyên nhân làm bảng dịch chuyển cụm bị tách dòng)
name_sets = {year: set(df['country'].astype(str).str.strip()) for year, df in raw_data.items()}
print('Số quốc gia mỗi năm:', {year: len(names) for year, names in name_sets.items()})
print('\nCó ở 2018 nhưng không có ở 2019:', sorted(name_sets[2018] - name_sets[2019]))
print('Có ở 2019 nhưng không có ở 2018:', sorted(name_sets[2019] - name_sets[2018]))
print('\nBảng đổi tên quốc gia đang áp dụng:', COUNTRY_NAME_FIXES)

# Quốc gia thiếu giá trị ở năm 2018
missing_2018 = raw_data[2018][raw_data[2018]['corruption_perception'].isna()][['country', 'corruption_perception']]
print('\nQuốc gia thiếu Perceptions of corruption năm 2018:')
display(missing_2018)

Số quốc gia mỗi năm: {2015: 158, 2016: 157, 2017: 155, 2018: 156, 2019: 156}

Có ở 2018 nhưng không có ở 2019: ['Angola', 'Belize', 'Macedonia', 'Sudan']
Có ở 2019 nhưng không có ở 2018: ['Comoros', 'Gambia', 'North Macedonia', 'Swaziland']

Bảng đổi tên quốc gia đang áp dụng: {'Macedonia': 'North Macedonia', 'Swaziland': 'Eswatini', 'Trinidad & Tobago': 'Trinidad and Tobago'}

Quốc gia thiếu Perceptions of corruption năm 2018:


,country,corruption_perception
19,United Arab Emirates,NaN


**Nhận xét bước 3:**
1. Cỡ mẫu khác nhau giữa các năm, nên mọi so sánh theo thời gian ở các bước sau phải ghi rõ cỡ mẫu.
2. Cột `region` chỉ tồn tại ở 2015-2016, còn `dystopia_residual` không có ở 2018-2019.
3. Năm 2018 thiếu 1 giá trị `Perceptions of corruption`; quốc gia đó sẽ bị loại khỏi thực nghiệm của năm 2018 và được ghi rõ trong phần hạn chế của báo cáo.
4. Một số quốc gia đổi tên giữa các năm, đã được chuẩn hóa trước khi ghép để tránh tách dòng.

## 4. Tiền xử lý và dựng bảng trung gian 5 năm
- Chuẩn hóa tên cột về snake_case, chuẩn hóa tên quốc gia.
- Chỉ giữ bộ cột trong `INTERIM_COLUMNS`; các cột độ bất định của điểm hạnh phúc bị loại bỏ.
- Ghép 5 năm thành một bảng có schema thống nhất và lưu vào `data/interim/happiness_merged.csv`.

Các cột bị loại bỏ và lý do:

In [5]:
for column, reason in DROPPED_COLUMNS_NOTE.items():
    print(f'- {column}: {reason}')

print('\nCác cột còn lại của bảng trung gian:')
print(INTERIM_COLUMNS)

- Standard Error: sai số chuẩn của điểm hạnh phúc, không phải yếu tố cơ sở
- Lower Confidence Interval: khoảng tin cậy của điểm hạnh phúc, không phải yếu tố cơ sở
- Upper Confidence Interval: khoảng tin cậy của điểm hạnh phúc, không phải yếu tố cơ sở
- Whisker.high: biên trên khoảng bất định của điểm hạnh phúc ở bản 2017
- Whisker.low: biên dưới khoảng bất định của điểm hạnh phúc ở bản 2017

Các cột còn lại của bảng trung gian:
['year', 'country', 'region', 'happiness_rank', 'happiness_score', 'gdp_per_capita', 'social_support', 'healthy_life_expectancy', 'freedom', 'generosity', 'corruption_perception', 'dystopia_residual']


In [6]:
interim_path = project_root / 'data' / 'interim' / 'happiness_merged.csv'
interim_df = build_interim_dataset(
    years=DEFAULT_YEARS,
    raw_dir=project_root / 'data' / 'raw',
    output_path=interim_path,
)

print('Đã lưu bảng trung gian:', interim_path)
print('Kích thước bảng:', interim_df.shape)
print('\nSố dòng theo năm:')
print(interim_df['year'].value_counts().sort_index())
print('\nSố giá trị thiếu theo cột (chỉ in cột có thiếu):')
missing = interim_df.isna().sum()
print(missing[missing > 0])
interim_df.head()

Đã lưu bảng trung gian: C:\Users\xpaga\Documents\data_mining\data\interim\happiness_merged.csv
Kích thước bảng: (782, 12)

Số dòng theo năm:
year
2015    158
2016    157
2017    155
2018    156
2019    156
Name: count, dtype: int64

Số giá trị thiếu theo cột (chỉ in cột có thiếu):
region                   467
corruption_perception      1
dystopia_residual        312
dtype: int64


,year,country,region,happiness_rank,happiness_score,gdp_per_capita,social_support,healthy_life_expectancy,freedom,generosity,corruption_perception,dystopia_residual
0,2015,Afghanistan,Southern Asia,153,3.575,0.31982,0.30285,0.30335,0.23414,0.36510,0.09719,1.9521
1,2015,Albania,Central and Eastern Europe,95,4.959,0.87867,0.80434,0.81325,0.35733,0.14272,0.06413,1.89894
2,2015,Algeria,Middle East and Northern Africa,68,5.605,0.93929,1.07772,0.61766,0.28579,0.07822,0.17383,2.43209
3,2015,Angola,Sub-Saharan Africa,137,4.033,0.75778,0.86040,0.16683,0.10384,0.12344,0.07122,1.94939
4,2015,Argentina,Latin America and Caribbean,30,6.574,1.05351,1.24823,0.78723,0.44974,0.11451,0.08484,2.836


In [7]:
# Trích 10 dòng đầu để chèn vào báo cáo Word
preview_path = tables_dir / 'interim_preview.csv'
interim_df.head(10).to_csv(preview_path, index=False, encoding='utf-8-sig')
print('Đã lưu trích dẫn cho báo cáo:', preview_path)
interim_df.head(10)

Đã lưu trích dẫn cho báo cáo: C:\Users\xpaga\Documents\data_mining\reports\tables\interim_preview.csv


,year,country,region,happiness_rank,happiness_score,gdp_per_capita,social_support,healthy_life_expectancy,freedom,generosity,corruption_perception,dystopia_residual
0,2015,Afghanistan,Southern Asia,153,3.575,0.31982,0.30285,0.30335,0.23414,0.36510,0.09719,1.9521
1,2015,Albania,Central and Eastern Europe,95,4.959,0.87867,0.80434,0.81325,0.35733,0.14272,0.06413,1.89894
2,2015,Algeria,Middle East and Northern Africa,68,5.605,0.93929,1.07772,0.61766,0.28579,0.07822,0.17383,2.43209
3,2015,Angola,Sub-Saharan Africa,137,4.033,0.75778,0.86040,0.16683,0.10384,0.12344,0.07122,1.94939
4,2015,Argentina,Latin America and Caribbean,30,6.574,1.05351,1.24823,0.78723,0.44974,0.11451,0.08484,2.836
5,2015,Armenia,Central and Eastern Europe,127,4.350,0.76821,0.77711,0.72990,0.19847,0.07855,0.03900,1.75873
6,2015,Australia,Australia and New Zealand,10,7.284,1.33358,1.30923,0.93156,0.65124,0.43562,0.35637,2.26646
7,2015,Austria,Western Europe,13,7.200,1.33723,1.29704,0.89042,0.62433,0.33088,0.18676,2.5332
8,2015,Azerbaijan,Central and Eastern Europe,80,5.212,1.02389,0.93793,0.64045,0.37030,0.07799,0.16065,2.00073
9,2015,Bahrain,Middle East and Northern Africa,49,5.960,1.32376,1.21624,0.74716,0.45492,0.17362,0.30600,1.73797


### Kiểm tra quy tắc vàng của đề tài
Ô dưới đây chặn việc vô tình đưa điểm hạnh phúc hoặc thứ hạng hạnh phúc vào ma trận đặc trưng dùng để gom cụm.

In [8]:
FORBIDDEN_IN_FEATURES = {'happiness_score', 'happiness_rank'}
assert FORBIDDEN_IN_FEATURES.isdisjoint(FEATURE_COLUMNS), (
    'Vi phạm quy tắc vàng: điểm hoặc thứ hạng hạnh phúc nằm trong ma trận đặc trưng'
)
print('Ma trận đặc trưng dùng để gom cụm:', FEATURE_COLUMNS)
print('Đã kiểm tra: không chứa', sorted(FORBIDDEN_IN_FEATURES))
print('Hai thuộc tính này chỉ được dùng để hậu kiểm ở notebook 04.')

Ma trận đặc trưng dùng để gom cụm: ['gdp_per_capita', 'social_support', 'healthy_life_expectancy', 'freedom', 'generosity', 'corruption_perception']
Đã kiểm tra: không chứa ['happiness_rank', 'happiness_score']
Hai thuộc tính này chỉ được dùng để hậu kiểm ở notebook 04.


## Kết luận notebook 01 và bước tiếp theo

**Đã hoàn thành:** bước 1, 2, 3 và phần 1 của bước 4 trong pipeline 11 bước.

**Đầu ra:**
- `data/interim/happiness_merged.csv`
- `reports/tables/data_quality_by_year.csv`
- `reports/tables/interim_preview.csv`

**Notebook tiếp theo:** `02_exploratory_data_analysis.ipynb` — EDA có mục tiêu (phân phối, tương quan, ngoại lệ) và chốt lựa chọn cách chuẩn hóa cho bước 6.